<a href="https://colab.research.google.com/github/Teomorales20/SenalesSistemas/blob/master/Copia_de_se%C3%B1ales_peri%C3%B3dicas_y_aperi%C3%B3dicas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros de la señal
T = 1.0                   # periodo fundamental
omega0 = 2 * np.pi / T    # frecuencia angular fundamental
k = 2                     # armónico elegido (entero)
t0 = 0.1                  # desplazamiento en el tiempo

# Definición de coeficientes de Fourier de x(t) = exp(i k w0 t)
def fourier_coeff_x(n):
    return 1.0 if n == k else 0.0

# Coeficientes después del desplazamiento: y(t) = x(t - t0)
def fourier_coeff_y(n):
    return fourier_coeff_x(n) * np.exp(-1j * n * omega0 * t0)

# Rango de índices n para graficar
n_vals = np.arange(-5, 6)
x_coeffs = np.array([fourier_coeff_x(n) for n in n_vals])
y_coeffs = np.array([fourier_coeff_y(n) for n in n_vals])

# --- Gráficas ---
plt.figure(figsize=(12,4))

# Magnitud de coeficientes antes/después
plt.subplot(1,2,1)
plt.stem(n_vals, np.abs(x_coeffs), basefmt=" ", linefmt="C0-", markerfmt="C0o", label="|x_n| (original)")
plt.stem(n_vals, np.abs(y_coeffs), basefmt=" ", linefmt="C1-", markerfmt="C1s", label="|y_n| (desplazada)")
plt.title("Magnitud de los coeficientes")
plt.xlabel("n")
plt.ylabel("|coef|")
plt.legend()
plt.grid(True)

# Fase de coeficientes antes/después
plt.subplot(1,2,2)
plt.stem(n_vals, np.angle(x_coeffs), basefmt=" ", linefmt="C0-", markerfmt="C0o", label="∠x_n")
plt.stem(n_vals, np.angle(y_coeffs), basefmt=" ", linefmt="C1-", markerfmt="C1s", label="∠y_n")
plt.title("Fase de los coeficientes (rad)")
plt.xlabel("n")
plt.ylabel("Fase [rad]")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Mostrar el valor del único coeficiente distinto de cero
print(f"Coeficiente no nulo (n={k}):")
print(f"x_{k} = {fourier_coeff_x(k)} (fase {np.angle(fourier_coeff_x(k)):.3f} rad)")
print(f"y_{k} = {fourier_coeff_y(k)} (fase {np.angle(fourier_coeff_y(k)):.3f} rad)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

print("matplotlib version:", matplotlib.__version__)  # útil para depurar

# Parámetros de ejemplo (usa los tuyos)
tau = 1.0
T = 2.0
w0 = np.pi
t0 = 0.25
N = 15

# rect periódica (igual que antes)
def rect_pulse(t, tau=1.0, T=2.0, t0=0.0):
    t_mod = ((t - t0 + T/2) % T) - T/2
    return np.where(np.abs(t_mod) <= tau/2, 1.0, 0.0)

def fourier_coeffs(N, t0, tau=1.0, T=2.0):
    coeffs = []
    for n in range(1, N+1):
        c_n = np.sin(n*np.pi/2)/(n*np.pi)
        a_n = 2*c_n*np.cos(n*np.pi*t0)
        b_n = 2*c_n*np.sin(n*np.pi*t0)
        coeffs.append((a_n, b_n))
    return coeffs

def reconstruct(t, N, t0):
    s = 0.5
    coeffs = fourier_coeffs(N, t0)
    for n, (a_n, b_n) in enumerate(coeffs, start=1):
        s += a_n*np.cos(n*np.pi*t) + b_n*np.sin(n*np.pi*t)
    return s

# Plots
t = np.linspace(-2, 2, 2000)
original = rect_pulse(t, tau, T, t0)
reconstructed = reconstruct(t, N, t0)

plt.figure(figsize=(10,5))
plt.plot(t, original, lw=2, label='Pulso original (desplazado)')
plt.plot(t, reconstructed, '--', lw=2, label=f'Serie de Fourier (N={N})')
plt.title(f"Pulso rectangular desplazado (t0={t0})")
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.legend()
plt.show()

# Coeficientes
coeffs = fourier_coeffs(N, t0)
a_vals = [a for a, b in coeffs]
b_vals = [b for a, b in coeffs]
n_vals = np.arange(1, N+1)

plt.figure(figsize=(12,4))
# <-- aquí quité use_line_collection=True -->
plt.stem(n_vals, a_vals, basefmt=" ", markerfmt="o", linefmt="-")
plt.title("Coeficientes a_n (coseno)")
plt.xlabel("n")
plt.ylabel("a_n")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# tu f(t,T) permanece igual
def f(t,T):
    T_local = 3 * np.pi  # Fundamental period of the signal
    t_mod = ((t + T_local / 2) % T_local) - T_local / 2
    return np.where(np.abs(t_mod) <= np.pi / 4, 1.0, 0.0)

# parámetros
T = 3 * np.pi
omega0 = 2 * np.pi / T
# el desplazamiento que elimina los a_n
t0 = 3 * np.pi / 4   # ≈ 2.35619

# función para coeficientes (igual que la tuya)
def compute_fourier_coefficients(f, T, N):
    a0, _ = quad(lambda x: f(x,T), 0, T)
    a0 = a0 / T
    an = []
    bn = []
    for n in range(1, N + 1):
        an_val, _ = quad(lambda x: f(x,T) * np.cos(2 * np.pi * n * x / T), 0, T)
        an.append(2 * an_val / T)
        bn_val, _ = quad(lambda x: f(x,T) * np.sin(2 * np.pi * n * x / T), 0, T)
        bn.append(2 * bn_val / T)
    return a0, an, bn

# calcular coeficientes con el t0 elegido (si quieres desplazar la señal en el tiempo,
# deberías integrar f(x-t0) o mover f dentro de la llamada; aquí ejemplo sencillo:
# reconstrucción con t0 incorporado en la reconstrucción trigonométrica)
N = 21
a0, an, bn = compute_fourier_coefficients(lambda x, T: f(x - t0, T), T, N)

# mostrar primeras componentes (ver que an ≈ 0)
for n in range(1, 11):
    print(f"n={n:2d}  a_n={an[n-1]: .3e}    b_n={bn[n-1]: .3e}")

# graficas
t = np.linspace(-T, T, 2048)
y_original = f(t - t0, T)  # original desplazada
# reconstrucción trigonométrica
def fourier_series(t, a0, an, bn, T, N):
    result = a0 * np.ones_like(t)
    for n in range(N):
        result += an[n] * np.cos(2 * np.pi * (n + 1) * t / T) + bn[n] * np.sin(2 * np.pi * (n + 1) * t / T)
    return result

y_fourier = fourier_series(t, a0, an, bn, T, N)

plt.figure(figsize=(10,4))
plt.plot(t, y_original, label='Original desplazada', linewidth=2)
plt.plot(t, y_fourier, '--', label=f'Reconstrucción N={N}')
plt.legend(); plt.grid(True); plt.title(f"t0 = {t0:.4f}")
plt.show()

# graficar coeficientes (usa stem sin argumentos problemáticos)
n_values = np.arange(1, N + 1)
plt.figure(figsize=(10,3))
plt.stem(n_values, an, markerfmt='bo', linefmt='b-')
plt.title('a_n (debe ser ~0)')
plt.show()

plt.figure(figsize=(10,3))
plt.stem(n_values, bn, markerfmt='go', linefmt='g-')
plt.title('b_n (deben ser los no nulos)')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# ------------------------
# Definir señal periódica (pulso rectangular)
# ------------------------
def f(t,T):
    T = 2*np.pi  # periodo base
    t_mod = ((t + T/2) % T) - T/2  # mapear a [-T/2, T/2)
    return np.where(np.abs(t_mod) <= np.pi/2, 1.0, 0.0)

# ------------------------
# Coeficientes de Fourier
# ------------------------
def compute_fourier_coefficients(f, T, N):
    a0, _ = quad(lambda x: f(x,T), 0, T)
    a0 = a0 / T
    an, bn = [], []
    for n in range(1, N + 1):
        an_val, _ = quad(lambda x: f(x,T) * np.cos(2 * np.pi * n * x / T), 0, T)
        an.append(2 * an_val / T)
        bn_val, _ = quad(lambda x: f(x,T) * np.sin(2 * np.pi * n * x / T), 0, T)
        bn.append(2 * bn_val / T)
    return a0, an, bn

# ------------------------
# Reconstrucción Fourier
# ------------------------
def fourier_series(t, a0, an, bn, T, N):
    result = a0 * np.ones_like(t)
    for n in range(N):
        result += an[n] * np.cos(2 * np.pi * (n+1) * t / T) + bn[n] * np.sin(2 * np.pi * (n+1) * t / T)
    return result

# ------------------------
# Parámetros
# ------------------------
T = 2*np.pi
Nmax = 50
a0, an, bn = compute_fourier_coefficients(f, T, Nmax)

# ------------------------
# Graficar Gibbs con diferentes N
# ------------------------
t = np.linspace(-2*np.pi, 2*np.pi, 3000)  # más puntos para ver oscilaciones
y_original = f(t,T)

plt.figure(figsize=(12,6))
plt.plot(t, y_original, 'k', linewidth=2, label='Original')

for N in [1, 5, 10, 50]:
    yN = fourier_series(t, a0, an, bn, T, N)
    plt.plot(t, yN, '--', linewidth=1.5, label=f'N={N}')

plt.xlim([-2*np.pi, 2*np.pi])
plt.ylim([-0.5, 1.5])
plt.title('Gibbs Phenomenon in Fourier Series Approximation')
plt.xlabel('t')
plt.ylabel('f(t)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Señal periódica (onda cuadrada)
T = 2*np.pi
t = np.linspace(-2*T, 2*T, 2000)
x_periodica = np.sign(np.sin(t))   # onda cuadrada
